# 06. 비전 트랜스포머, 객체 탐지와 추적

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_06_%EB%B9%84%EC%A0%84_%ED%8A%B8%EB%9E%9C%EC%8A%A4%ED%8F%AC%EB%A8%B8_%EA%B0%9D%EC%B2%B4_%ED%83%90%EC%A7%80%EC%99%80_%EC%B6%94%EC%A0%81.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 패치 임베딩: 2차원 영상을 1차원 토큰 열로 바꾸기

### 🔬 실습 — PyTorch로 패치 임베딩의 모양 확인하기


In [ ]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16,
                 in_channels=3, embed_dim=192):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError("image_size must be divisible by patch_size")
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size,
                              stride=patch_size)

    def forward(self, x):
        # (B,C,H,W) -> (B,D,H/P,W/P) -> (B,N,D)
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)

patcher = PatchEmbedding(image_size=224, patch_size=16,
                         in_channels=3, embed_dim=192)
x = torch.randn(8, 3, 224, 224)
tokens = patcher(x)
print(tokens.shape, patcher.num_patches)

**실행 결과 예시**

```
torch.Size([8, 196, 192]) 196
```


## ViT encoder 블록과 분류 헤드

### 🔬 실습 — 작은 ViT를 직접 구성해 한 번 순전파하기

다음 코드는 교육용 최소 구조다. 실제 모델보다 작지만 패치 임베딩, `[CLS]`, 위치 임베딩, Transformer encoder, 분류 헤드의 연결을 모두 포함한다.


In [ ]:
import torch
import torch.nn as nn

class TinyViT(nn.Module):
    def __init__(self, image_size=64, patch_size=8,
                 embed_dim=128, depth=3, heads=4,
                 num_classes=6):
        super().__init__()
        self.patch = nn.Conv2d(3, embed_dim,
                               kernel_size=patch_size,
                               stride=patch_size)
        n = (image_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x = self.patch(x).flatten(2).transpose(1, 2)  # (B,N,D)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.encoder(x)
        return self.head(self.norm(x[:, 0]))

model = TinyViT()
images = torch.randn(4, 3, 64, 64)
logits = model(images)
print(logits.shape)

**실행 결과 예시**

```
torch.Size([4, 6])
```


## 사전학습 ViT 전이학습과 운용 점검

### 🔬 실습 — 사전학습 ViT의 분류 헤드 교체하기


In [ ]:
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

num_classes = 6
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights)

# 1단계: 백본을 동결하고 새 헤드만 학습
for p in model.parameters():
    p.requires_grad = False
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, num_classes)

# 사전학습과 동일한 resize/normalize 변환
preprocess = weights.transforms()

# 2단계: 검증 성능이 정체되면 뒤쪽 encoder block 일부를 해제
for p in model.encoder.layers[-2:].parameters():
    p.requires_grad = True

## 실습: NMS 구현과 사전학습 탐지 모델 추론

### 🔬 실습 — NMS 직접 구현과 Faster R-CNN 추론

① **NMS 구현** — 정렬 후 최고점 박스를 남기고 IoU 임계값 이상을 제거.


In [ ]:
import torch

def iou(box, boxes):                       # box:(4,), boxes:(N,4) (x1,y1,x2,y2)
    x1 = torch.max(box[0], boxes[:, 0]); y1 = torch.max(box[1], boxes[:, 1])
    x2 = torch.min(box[2], boxes[:, 2]); y2 = torch.min(box[3], boxes[:, 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    a  = (box[2]-box[0]) * (box[3]-box[1])
    ab = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
    return inter / (a + ab - inter + 1e-9)

def nms(boxes, scores, thr=0.5):
    keep, idx = [], scores.argsort(descending=True)   # 1) 점수 내림차순
    while idx.numel() > 0:
        i = idx[0]; keep.append(i.item())             # 2) 최고점 박스 채택
        if idx.numel() == 1: break
        ious = iou(boxes[i], boxes[idx[1:]])
        idx = idx[1:][ious < thr]                      # 3) IoU<thr 만 남김
    return keep

② **사전학습 Faster R-CNN 추론** — torchvision의 COCO 사전학습 모델 사용.


In [ ]:
import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights).eval()   # 평가 모드
img = torch.rand(3, 480, 640)                              # 더미 입력 (C,H,W)
with torch.no_grad():
    out = model([img])[0]      # dict: 'boxes','labels','scores'
keep = out["scores"] > 0.5     # 신뢰도 0.5 이상만
print("탐지 수:", keep.sum().item())

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
